In [1]:
import sys
import logging
from pathlib import Path
import time

import numpy as np
import pandas as pd
import duckdb
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.model_selection import train_test_split

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)

PROJECT_ROOT = Path.cwd().parent
SRC_PATH     = PROJECT_ROOT / "src"
DATA_PATH    = PROJECT_ROOT / "data" / "processed" / "train.parquet"
MODEL_PATH   = PROJECT_ROOT / "models" / "sklearn"

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

from trainers import sklearn_trainer as skt
from utils import evaluation_toolkit as eva

con = duckdb.connect(database=":memory:")
con.execute(
    f"CREATE VIEW train AS SELECT * FROM read_parquet('{DATA_PATH.as_posix()}')"
)
con.execute("PRAGMA disable_progress_bar")

# Entrenamiento del Perceptrón con Sklearn

:::{admonition} Resumen
:class: info
- La búsqueda de hiperparámetros se ejecutó mediante `run_grid_search_manual_kfold` con cinco folds estratificados, métrica objetivo AUC y una duración aproximada de 3 horas.
- El suavizado de la codificación objetivo en diez mejoró el rendimiento del clasificador sobre la submuestra.
- El reentrenamiento del mejor modelo sobre la muestra completa alcanzó un AUC de .7493 en validación, superior al .7457 de la búsqueda.
- El test de DeLong no detectó diferencia significativa entre la arquitectura de una y dos capas ocultas ($p = .485$).
- Se retuvo el clasificador de una capa oculta de cien unidades por parsimonia y menor costo de inferencia.
:::

## Introducción

El Click-Through Rate (CTR) mide la proporción de impresiones publicitarias que resultan en un clic por parte del usuario. La estimación de esta probabilidad permite optimizar la asignación de presupuesto publicitario y personalizar la entrega de anuncios. El modelado se apoya en clasificadores capaces de operar sobre variables categóricas de alta cardinalidad.

:::{admonition} Nota: Perceptrón Multicapa
:class: note

El Perceptrón Multicapa (MLP) es una red neuronal feedforward compuesta por una capa de entrada, una o más capas ocultas y una capa de salida. Cada neurona aplica una transformación afín seguida de una función de activación no lineal. El entrenamiento ajusta los pesos de la red mediante descenso de gradiente sobre una función de pérdida. La arquitectura es adecuada para problemas de clasificación binaria con variables de entrada heterogéneas.
:::

El conjunto de entrenamiento del corpus Avazu contiene millones de impresiones. El entrenamiento del clasificador sobre la totalidad del conjunto es computacionalmente costoso. Esta libreta extrae una submuestra aleatoria estratificada de un millón de registros sobre la variable de respuesta ``click``. El muestreo preserva la proporción de clases del conjunto original y la reproducibilidad queda garantizada por una semilla fija.

In [2]:
df_sample = skt.get_stratified_sample(
    con=con, relation="train", n=1_000_000,
    target="click",
    set_labels=("Conjunto original", "Submuestra"),
    column_titles=(
        "Conjunto",
        "Registros",
        "Proporción de clics",
        "Δ Proporción",
    )
)

Conjunto,Registros,Proporción de clics,Δ Proporción
Conjunto original,"28,294,671",.1698,—
Submuestra,"1,000,000",.1698,.0000


## Entrenamiento

Para el entrenamiento se construyó un pipeline que aplicó la estrategia de codificación definida en el análisis exploratorio. El transformador ``AvazuPreprocessor`` dicotomizó las variables de categoría dominante, aplicó codificación top-K con categoría residual y calculó tasas objetivo suavizadas. El transformador ``SelectiveScaler`` estandarizó únicamente las columnas numéricas de frecuencia. El clasificador final fue un ``MLPClassifier``.

La búsqueda exploró de forma conjunta la profundidad de la red y el suavizado de la codificación objetivo. La evaluación simultánea permitió medir la interacción entre la capacidad de representación y la calidad de las variables codificadas. El resultado aisló el efecto del suavizado sobre el rendimiento del clasificador.

In [3]:
X = df_sample.drop(columns=["click"])
y = df_sample["click"]

pipe = make_pipeline(
    skt.AvazuPreprocessor(smoothing=10),
    skt.SelectiveScaler(indices=(3, 4)),
    MLPClassifier(
        random_state=42,
        early_stopping=True,
        n_iter_no_change=10,
        validation_fraction=0.2,
        tol=1e-4,
    ),
)

param_grid = {
    "avazupreprocessor__smoothing": [0, 10],
    "mlpclassifier__hidden_layer_sizes": [(50,), (100,), (100, 50)],
    "mlpclassifier__alpha": [0.0001, 0.001, 0.01],
    "mlpclassifier__max_iter": [100, 200, 500]
}

La búsqueda se ejecutó mediante la función ``run_grid_search_manual_kfold``, que implementa una validación cruzada estratificada con escritura incremental de los resultados a un archivo JSON. El registro por combinación permitió monitorear el avance de la búsqueda y reanudarla tras cualquier interrupción sin repetir combinaciones evaluadas.

La métrica objetivo fue el área bajo la curva ROC. El clasificador de la librería no implementa pesos de clase para compensar el desbalance, por lo que la selección de umbral quedó fuera del alcance del ajuste. El AUC evalúa el ordenamiento de las probabilidades con independencia del punto de corte. Métricas como el F1-score dependen de un umbral de decisión y no se consideraron en esta fase.

In [28]:
df_results = skt.run_grid_search_manual_kfold(
    pipe=pipe,
    param_grid=param_grid,
    X_train=X_train,
    y_train=y_train,
    model_path=MODEL_PATH,
    cv=5,
    scoring="roc_auc",
    results_filename="grid_sklearn.json"
)

INFO:__main__:Grid search: 54 combinations, cv=5
GridSearch: 100%|██████████| 54/54 [2:48:49<00:00, 187.58s/combo, mean AUC=0.7448]      
INFO:__main__:Total wall time: 168.82 min
INFO:__main__:Best AUC: 0.7457
INFO:__main__:Best params: {'avazupreprocessor__smoothing': 10, 'mlpclassifier__hidden_layer_sizes': (100, 50), 'mlpclassifier__alpha': 0.001, 'mlpclassifier__max_iter': 200}


La búsqueda concluyó sin incidencias en aproximadamente 2.8 horas. La diferencia entre la arquitectura de una capa oculta y la de dos capas ocultas fue de cuatro diezmilésimas en el AUC medio. La regularización débil fue suficiente para alcanzar el mejor rendimiento, lo que indica que la red no requirió una restricción fuerte sobre la magnitud de los pesos.

Las tres configuraciones de número máximo de iteraciones produjeron resultados idénticos dentro de cada combinación de arquitectura y regularización. El mecanismo de detención temprana se activó antes de la centésima iteración en todas las combinaciones evaluadas. El costo computacional del ajuste quedó determinado por la arquitectura y el tamaño de la submuestra.

In [4]:
_ = skt.grid_results_table(
    MODEL_PATH / "grid_sklearn.json",
    top_n=10,
    param_display_names={
        "mlpclassifier__hidden_layer_sizes": "Capas ocultas",
        "mlpclassifier__alpha": "Alfa",
        "mlpclassifier__max_iter": "Máx. iter",
        "avazupreprocessor__smoothing": "Suavizado",
    },
    auc_title="AUC",
)

Suavizado,Capas ocultas,Alfa,Máx. iter,AUC
10,"(100, 50)",0.001,100,0.7457 ± 0.0016
10,"(100, 50)",0.001,200,0.7457 ± 0.0016
10,"(100, 50)",0.001,500,0.7457 ± 0.0016
10,"(100, 50)",0.0001,100,0.7457 ± 0.0011
10,"(100, 50)",0.0001,200,0.7457 ± 0.0011
10,"(100, 50)",0.0001,500,0.7457 ± 0.0011
10,(100),0.0001,100,0.7451 ± 0.0016
10,(100),0.0001,200,0.7451 ± 0.0016
10,(100),0.0001,500,0.7451 ± 0.0016
0,"(100, 50)",0.0001,100,0.7450 ± 0.0015


El mejor modelo identificado en la búsqueda se reentrenó sobre la muestra completa con la partición estratificada por la variable de respuesta. El reentrenamiento empleó el entrenador iterativo ``MLPTrainerWithTracking``, que expone la evolución de la pérdida y el AUC por época. La paciencia y el umbral de mejora mínima se relajaron respecto a los valores por defecto del clasificador, con el fin de no detener el ajuste antes de la estabilización de las curvas.

In [5]:
pipe = make_pipeline(
    skt.AvazuPreprocessor(smoothing=10),
    skt.SelectiveScaler(indices=(3, 4)),
    MLPClassifier(
        hidden_layer_sizes=(100, 50),
        alpha=0.001,
        random_state=42
    )
)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

trainer = skt.MLPTrainerWithTracking(
    pipeline=pipe,
    epochs=500,
    patience=25,
    min_delta=1e-6
)

start = time.perf_counter()
df_history = trainer.fit(X_train, y_train, X_val, y_val)
training_time_1 = time.perf_counter() - start

Transforming data with the preprocessing pipeline...
Starting iterative training for 500 epochs...
Epoch 001/500 | Train Loss: 0.3987 - Val Loss: 0.4016 | Train AUC: 0.7480 - Val AUC: 0.7421
Epoch 005/500 | Train Loss: 0.3952 - Val Loss: 0.3990 | Train AUC: 0.7530 - Val AUC: 0.7464
Epoch 010/500 | Train Loss: 0.3931 - Val Loss: 0.3974 | Train AUC: 0.7557 - Val AUC: 0.7481
Epoch 015/500 | Train Loss: 0.3927 - Val Loss: 0.3976 | Train AUC: 0.7566 - Val AUC: 0.7483
Epoch 020/500 | Train Loss: 0.3923 - Val Loss: 0.3974 | Train AUC: 0.7571 - Val AUC: 0.7484
Epoch 025/500 | Train Loss: 0.3920 - Val Loss: 0.3974 | Train AUC: 0.7578 - Val AUC: 0.7485
Epoch 030/500 | Train Loss: 0.3915 - Val Loss: 0.3973 | Train AUC: 0.7587 - Val AUC: 0.7490
Epoch 035/500 | Train Loss: 0.3913 - Val Loss: 0.3971 | Train AUC: 0.7589 - Val AUC: 0.7490
Epoch 040/500 | Train Loss: 0.3913 - Val Loss: 0.3975 | Train AUC: 0.7591 - Val AUC: 0.7489
Epoch 045/500 | Train Loss: 0.3914 - Val Loss: 0.3977 | Train AUC: 0.7592

El AUC sobre la muestra de validación fue de 0.749, superior al 0.746 obtenido en la búsqueda de hiperparámetros ($\Delta \text{AUC} =$ 0.004). Las curvas de entrenamiento y validación presentaron una separación relativamente estable y mostraron una tendencia decreciente en la pérdida de validación sin reversión al final del entrenamiento.

In [13]:
_, _ = skt.plot_learning_curves_pair(
    df_history,
    titles=("Descenso de Gradiente (Log Loss)", "Evolución del AUC"),
    xlabels=("Época", "Época"),
    ylabels=("Log Loss", "ROC AUC"),
    legend_labels=("Entrenamiento", "Validación"),
    ylims=((0.39, 0.42), (0.74, 0.78)),
    y_steps=(0.005, 0.008),
    show_legend=(True, False),
    legend_frame=False, legend_loc="upper left",
    legend_bbox=(0.05, 0.95),
    pads=(10, 10, 12), label_size=11, tick_size=10,
)

El modelo con una sola capa oculta de cien neuronas alcanzó un AUC de 0.749 en la muestra de validación, un rendimiento comparable alcanzado por el modelo con dos capas ocultas. El número de épocas necesarias para la estabilización fue mayor que en la arquitectura de una sola capa. La curva de aprendizaje del modelo menos complejo fue más suave que la del modelo más profundo.

In [6]:
pipe = make_pipeline(
    skt.AvazuPreprocessor(smoothing=10),
    skt.SelectiveScaler(indices=(3, 4)),
    MLPClassifier(
        hidden_layer_sizes=(100,),
        alpha=0.001,
        random_state=42,
    )
)

trainer2 = skt.MLPTrainerWithTracking(
    pipeline=pipe,
    epochs=500,
    patience=25,
    min_delta=1e-6,
)

start = time.perf_counter()
df_history_2 = trainer2.fit(X_train, y_train, X_val, y_val)
training_time_2 = time.perf_counter() - start

Transforming data with the preprocessing pipeline...
Starting iterative training for 500 epochs...
Epoch 001/500 | Train Loss: 0.3993 - Val Loss: 0.4024 | Train AUC: 0.7467 - Val AUC: 0.7408
Epoch 005/500 | Train Loss: 0.3966 - Val Loss: 0.4004 | Train AUC: 0.7524 - Val AUC: 0.7457
Epoch 010/500 | Train Loss: 0.3937 - Val Loss: 0.3977 | Train AUC: 0.7549 - Val AUC: 0.7476
Epoch 015/500 | Train Loss: 0.3931 - Val Loss: 0.3975 | Train AUC: 0.7560 - Val AUC: 0.7482
Epoch 020/500 | Train Loss: 0.3930 - Val Loss: 0.3975 | Train AUC: 0.7564 - Val AUC: 0.7482
Epoch 025/500 | Train Loss: 0.3928 - Val Loss: 0.3975 | Train AUC: 0.7569 - Val AUC: 0.7485
Epoch 030/500 | Train Loss: 0.3927 - Val Loss: 0.3975 | Train AUC: 0.7572 - Val AUC: 0.7486
Epoch 035/500 | Train Loss: 0.3927 - Val Loss: 0.3975 | Train AUC: 0.7572 - Val AUC: 0.7486
Epoch 040/500 | Train Loss: 0.3927 - Val Loss: 0.3976 | Train AUC: 0.7574 - Val AUC: 0.7487
Epoch 045/500 | Train Loss: 0.3927 - Val Loss: 0.3976 | Train AUC: 0.7575

La separación entre las curvas de entrenamiento y validación se mantuvo a pesar de la reducción en la complejidad del modelo. Este comportamiento sugiere que la red extrajo toda la información disponible en el conjunto de datos y que el aumento en la profundidad no aportó capacidad representacional adicional.

In [11]:
_, _ = skt.plot_learning_curves_pair(
    df_history_2,
    titles=("Descenso de Gradiente (Log Loss)", "Evolución del AUC"),
    xlabels=("Época", "Época"),
    ylabels=("Log Loss", "ROC AUC"),
    legend_labels=("Entrenamiento", "Validación"),
    ylims=((0.39, 0.42), (0.738, 0.78)),
    y_steps=(0.005, 0.008),
    show_legend=(True, False), legend_frame=False,
    legend_loc="upper left",
    legend_bbox=(0.05, 0.95), pads=(10, 10, 12),
    label_size=12, tick_size=11,
)

La diferencia en AUC entre ambas arquitecturas fue de dos diezmilésimas y no resultó estadísticamente significativa ($p = .485$). El modelo de dos capas ocultas requirió menos tiempo de entrenamiento pese a contener un mayor número de pesos por época debido al mecanismo de detención temprana. Los tiempos de predicción fueron idénticos, diferencia consistente con el costo de cómputo despreciable frente a los 200 000 registros de validación.

Dado que no existe evidencia de superioridad predictiva, el criterio de parsimonia favorece al modelo de una sola capa oculta. El modelo con menos parámetros reduce el riesgo de sobreajuste, simplifica la serialización y no incurre en costo operativo adicional.

In [8]:
fitted_pipeline_1 = Pipeline(
    steps=trainer.preprocessor.steps + [("mlpclassifier", trainer.mlp)]
)

fitted_pipeline_2 = Pipeline(
    steps=trainer2.preprocessor.steps + [("mlpclassifier", trainer2.mlp)]
)

start = time.perf_counter()
y_pred_proba_1 = fitted_pipeline_1.predict_proba(X_val)[:, 1]
prediction_time_1 = time.perf_counter() - start

start = time.perf_counter()
y_pred_proba_2 = fitted_pipeline_2.predict_proba(X_val)[:, 1]
prediction_time_2 = time.perf_counter() - start

_ = eva.delong_roc_table(
    y_true=y_val,
    y_pred1=y_pred_proba_1,
    y_pred2=y_pred_proba_2,
    model_names=("MLP (100, 50)", "MLP (100)"),
    training_times=(training_time_1, training_time_2),
    prediction_times=(prediction_time_1, prediction_time_2),
    column_titles={
        "Model": "Modelo",
        "Training Time": "Entrenamiento",
        "Prediction Time": "Predicción",
        "ΔAUC": "ΔAUC",
        "p": "p",
    },
)

Modelo,Entrenamiento,Predicción,AUC,ΔAUC,p
"MLP (100, 50)",669.45 s,2.71 s,.749,—,—
MLP (100),788.16 s,2.42 s,.749,-.000,.485


In [12]:
skt.save_sklearn_pipeline(
    trainer=trainer, model_dir=MODEL_PATH,
    pipeline_name="mlp_100_50_pipeline.joblib",
    history_name="training_MLP_100_50_history.json"
)

skt.save_sklearn_pipeline(
    trainer=trainer2, model_dir=MODEL_PATH,
    pipeline_name="mlp_100_pipeline.joblib",
    history_name="training_MLP_100_history.json"
)

2026-09-15 09:29:44 | INFO     | Pipeline saved to C:\Users\jcami\OneDrive\Escritorio\deep_learning\Avazu-CTR-DeepLearning\models\sklearn\mlp_100_50_pipeline.joblib
2026-09-15 09:29:44 | INFO     | History saved to C:\Users\jcami\OneDrive\Escritorio\deep_learning\Avazu-CTR-DeepLearning\models\sklearn\training_MLP_100_50_history.json
2026-09-15 09:29:45 | INFO     | Pipeline saved to C:\Users\jcami\OneDrive\Escritorio\deep_learning\Avazu-CTR-DeepLearning\models\sklearn\mlp_100_pipeline.joblib
2026-09-15 09:29:45 | INFO     | History saved to C:\Users\jcami\OneDrive\Escritorio\deep_learning\Avazu-CTR-DeepLearning\models\sklearn\training_MLP_100_history.json


{'pipeline_path': WindowsPath('C:/Users/jcami/OneDrive/Escritorio/deep_learning/Avazu-CTR-DeepLearning/models/sklearn/mlp_100_pipeline.joblib'),
 'history_path': WindowsPath('C:/Users/jcami/OneDrive/Escritorio/deep_learning/Avazu-CTR-DeepLearning/models/sklearn/training_MLP_100_history.json')}